<a href="https://colab.research.google.com/github/hernandezlorena616-bot/Call-Center/blob/main/Resultado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install simpy

In [4]:
import simpy
import random

# ==========================
# PARÁMETROS DE LA SIMULACIÓN
# ==========================
RANDOM_SEED = 123
TIEMPO_SIMULACION = 60        # Minutos
LLEGADA_PROMEDIO = 6          # Tiempo promedio entre llamadas
DURACION_MIN = 2              # Duración mínima de llamada
DURACION_MAX = 5              # Duración máxima de llamada
NUM_OPERADORES = 1

random.seed(RANDOM_SEED)

# ==========================
# VARIABLES PARA MÉTRICAS
# ==========================
clientes = 0
clientes_atendidos = 0
tiempo_espera = []
tiempo_llamada = []

# ==========================
# PROCESO DE ATENCIÓN
# ==========================
def atender_cliente(env, nombre, operador):
    global clientes_atendidos

    llegada = env.now

    print(f"{env.now:6.2f} min -> {nombre} llegó.")

    with operador.request() as request:
        yield request

        espera = env.now - llegada
        tiempo_espera.append(espera)

        print(f"{env.now:6.2f} min -> {nombre} es atendido. Esperó {espera:.2f} min")

        duracion = random.uniform(DURACION_MIN, DURACION_MAX)
        tiempo_llamada.append(duracion)

        yield env.timeout(duracion)

        clientes_atendidos += 1

        print(f"{env.now:6.2f} min -> {nombre} terminó la llamada ({duracion:.2f} min)")


# ==========================
# GENERADOR DE CLIENTES
# ==========================
def generar_clientes(env, operador):
    global clientes

    while True:
        tiempo = random.expovariate(1.0 / LLEGADA_PROMEDIO)

        yield env.timeout(tiempo)

        clientes += 1
        env.process(atender_cliente(env, f"Cliente {clientes}", operador))


# ==========================
# ENTORNO DE SIMULACIÓN
# ==========================
env = simpy.Environment()

operador = simpy.Resource(env, capacity=NUM_OPERADORES)

env.process(generar_clientes(env, operador))

env.run(until=TIEMPO_SIMULACION)

# ==========================
# MÉTRICAS FINALES
# ==========================
promedio_llamadas = sum(tiempo_llamada) / len(tiempo_llamada) if tiempo_llamada else 0
promedio_espera = sum(tiempo_espera) / len(tiempo_espera) if tiempo_espera else 0

print("\n")
print("---- FIN DE LA SIMULACIÓN ----")
print("=" * 45)
print("           MÉTRICAS FINALES")
print("=" * 45)
print(f"Tiempo de llamadas (Promedio): {promedio_llamadas:.2f} minutos")
print(f"Tiempo de espera (Promedio):   {promedio_espera:.2f} minutos")
print(f"Cantidad de clientes que llaman: {clientes}")
print(f"Cantidad de clientes atendidos:  {clientes_atendidos}")
print(f"Clientes pendientes:             {clientes-clientes_atendidos}")
print("=" * 45)

  0.32 min -> Cliente 1 llegó.
  0.32 min -> Cliente 1 es atendido. Esperó 0.00 min
  0.87 min -> Cliente 2 llegó.
  1.55 min -> Cliente 3 llegó.
  3.54 min -> Cliente 1 terminó la llamada (3.22 min)
  3.54 min -> Cliente 2 es atendido. Esperó 2.67 min
  5.66 min -> Cliente 2 terminó la llamada (2.11 min)
  5.66 min -> Cliente 3 es atendido. Esperó 4.11 min
  9.27 min -> Cliente 3 terminó la llamada (3.61 min)
 15.44 min -> Cliente 4 llegó.
 15.44 min -> Cliente 4 es atendido. Esperó 0.00 min
 17.86 min -> Cliente 5 llegó.
 18.91 min -> Cliente 6 llegó.
 20.00 min -> Cliente 4 terminó la llamada (4.56 min)
 20.00 min -> Cliente 5 es atendido. Esperó 2.13 min
 21.38 min -> Cliente 7 llegó.
 23.00 min -> Cliente 5 terminó la llamada (3.00 min)
 23.00 min -> Cliente 6 es atendido. Esperó 4.09 min
 23.06 min -> Cliente 8 llegó.
 25.00 min -> Cliente 6 terminó la llamada (2.01 min)
 25.00 min -> Cliente 7 es atendido. Esperó 3.63 min
 26.50 min -> Cliente 9 llegó.
 27.27 min -> Cliente 7 te